# 20 · Multi-plane aggregator
19b's aggregator reads ONE plane at a time; the 3 planes' answers are then simply averaged.
This one reads **all 3 planes together**: 3 × 24 = 72 slice "notes" per knee, each tagged with its slice position **and its plane**,
so it can learn e.g. "ACL → trust sagittal", "MCL → trust coronal", "PF OA → trust axial", and cross-check planes.

Same safety design as 19b: baseline + correction (starts equal to the plain average), feature noise, fixed 15 epochs, final weights (no picking by val), 5 seeds averaged.
Knees missing a plane are handled with a mask (the model just ignores that plane).

**Settings:** CPU is fine · Input: notebook `18_extract_features` only.

In [ ]:
# Cell 1: setup
import glob, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_auc_score

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
PLANES = ['sagittal', 'coronal', 'axial']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LO, HI = 6, 19
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 15

def find(name):
    f = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(f) == 1, (name, f)
    return f[0]
print(DEVICE)

In [ ]:
# Cell 2: regroup notebook 18's per-scan outputs into one row per KNEE with 3 plane slots
per = {}
for p in PLANES:
    ids = pd.read_csv(find(f'ids_{p}.csv'))
    per[p] = (np.load(find(f'feats_{p}.npy')), np.load(find(f'logits_{p}.npy')).astype(np.float32), ids)

knees = pd.concat([per[p][2][['StudyInstanceUID', 'split'] + LABELS] for p in PLANES]).drop_duplicates('StudyInstanceUID').reset_index(drop=True)
row_of = {s: i for i, s in enumerate(knees['StudyInstanceUID'])}
N = len(knees)

F = np.zeros((N, 3, 24, 512), np.float16)       # slice features, all planes
L = np.zeros((N, 3, 24, 12), np.float32)        # slice logits, all planes
M = np.zeros((N, 3), bool)                      # True = this knee has this plane
for j, p in enumerate(PLANES):
    f, l, ids = per[p]
    r = ids['StudyInstanceUID'].map(row_of).values
    F[r, j], L[r, j], M[r, j] = f, l, True
del per

Y = knees[LABELS].values.astype(np.float32)     # old labels (what model 14 trained on); val rows = real labels
tr = np.where(knees['split'] == 'train')[0]
va = np.where(knees['split'] == 'val')[0]
print('knees:', N, '| train', len(tr), '| val', len(va))
print('planes per knee:', pd.Series(M.sum(1)).value_counts().sort_index().to_dict())

In [ ]:
# Cell 3: scoring + the two numbers to beat
truth = knees.iloc[va].set_index('StudyInstanceUID')[LABELS] >= 0.5
sig = lambda x: 1 / (1 + np.exp(-x))

def knee_auc(prob_va):
    per = pd.Series({l: roc_auc_score(truth[l], prob_va[:, i]) for i, l in enumerate(LABELS)})
    return per.mean(), per

# baseline: mean of slices 6-18 per plane, then mean over the planes the knee has (= model 14 alone)
pp = sig(L[va, :, LO:HI]).mean(2)                                          # (n, 3, 12)
base_auc, base_per = knee_auc((pp * M[va, :, None]).sum(1) / M[va].sum(1, keepdims=True))
print(f'baseline (model 14, plane average) : {base_auc:.4f}   (should be ~0.788)')
print(f'19b per-plane aggregator, for reference: 0.8042')

In [ ]:
# Cell 4: the multi-plane aggregator
class MultiPlaneAgg(nn.Module):
    def __init__(self, d=128, drop=0.3):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(512), nn.Dropout(drop), nn.Linear(512, d), nn.GELU())
        self.pos = nn.Parameter(torch.zeros(1, 1, 24, d))      # which slice
        self.plane = nn.Parameter(torch.zeros(1, 3, 1, d))     # which plane
        layer = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=2 * d, dropout=drop, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=1, enable_nested_tensor=False)
        self.query = nn.Linear(d, 12)                          # per finding: which of the 72 slices to focus on
        self.out = nn.Linear(d, 12)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)   # correction starts at 0

    def forward(self, feats, logits, mask):                    # (B,3,24,512), (B,3,24,12), (B,3) bool
        B = feats.shape[0]
        x = (self.inp(feats) + self.pos + self.plane).reshape(B, 72, -1)
        pad = (~mask).repeat_interleave(24, dim=1)             # (B,72) True = missing plane -> ignore
        h = self.enc(x, src_key_padding_mask=pad)
        s = self.query(h).masked_fill(pad.unsqueeze(-1), -1e4)
        a = torch.softmax(s, dim=1)                            # (B,72,12)
        pooled = torch.einsum('bsf,bsd->bfd', a, h)            # (B,12,d)
        corr = (self.out.weight.unsqueeze(0) * pooled).sum(-1) + self.out.bias
        lp = logits[:, :, LO:HI].mean(2)                       # (B,3,12) baseline logit per plane
        m = mask.float().unsqueeze(-1)
        base = (lp * m).sum(1) / m.sum(1)                      # average over the planes the knee has
        return base + corr

In [ ]:
# Cell 5: training (fixed epochs, keep the FINAL weights)
def batches(idx, bs, shuffle):
    idx = np.random.permutation(idx) if shuffle else idx
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        yield (torch.from_numpy(F[j].astype(np.float32)).to(DEVICE), torch.from_numpy(L[j]).to(DEVICE),
               torch.from_numpy(M[j]).to(DEVICE), torch.from_numpy(Y[j]).to(DEVICE))

@torch.no_grad()
def predict(model, idx):
    model.eval()
    return np.concatenate([torch.sigmoid(model(f, l, m)).cpu().numpy() for f, l, m, _ in batches(idx, 128, False)])

def train(seed, lr=3e-4, epochs=EPOCHS):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MultiPlaneAgg().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    for ep in range(epochs):
        model.train(); tot = 0
        for f, l, m, y in batches(tr, 64, True):
            f = f + 0.1 * torch.randn_like(f)                  # feature noise (too-clean train features)
            loss = Fn.binary_cross_entropy_with_logits(model(f, l, m), y)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(y)
        if ep % 5 == 0 or ep == epochs - 1:
            print(f'   epoch {ep:2d}  loss {tot / len(tr):.4f}  val {knee_auc(predict(model, va))[0]:.4f}', flush=True)
    return model

In [ ]:
# Cell 6: 5 seeds, averaged
ps, ms = [], []
for s in SEEDS:
    m = train(s)
    p = predict(m, va); ps.append(p); ms.append(m)
    print(f'seed {s}: final val {knee_auc(p)[0]:.4f}')
mp_auc, mp_per = knee_auc(np.mean(ps, 0))
print(f'\nbaseline (model 14)           : {base_auc:.4f}')
print(f'19b per-plane aggregator       : 0.8042  (reference)')
print(f'multi-plane aggregator (5 avg) : {mp_auc:.4f}   change vs baseline: {mp_auc - base_auc:+.4f}')

In [ ]:
# Cell 7: per-finding table + save weights
print(pd.DataFrame({'baseline': base_per, 'multi-plane': mp_per}).round(3))
for s, m in zip(SEEDS, ms):
    torch.save(m.state_dict(), f'/kaggle/working/mpagg_seed{s}.pt')
print('saved:', sorted(glob.glob('/kaggle/working/mpagg_*.pt')))